In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
from pathlib import Path

# Корень репозитория и каталог данных — не зависят от того, откуда запущен
# ноутбук. commom_utils — namespace-пакет (без __init__.py), поэтому берём
# __path__, а не __file__: у таких пакетов __file__ равен None.
import commom_utils
REPO = Path(list(commom_utils.__path__)[0]).parent
DATASETS = REPO / "experiments" / "datasets"

from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
from gauss_newton.problem import MultipleShooting
from gauss_newton.adaptive import run_optimization_adaptive
from gauss_newton.collocation_shooting import CollocationShooting
from typing import Callable
from experiments.data_utils import LogReaderV2, create_interval_batches, theta_to_physical

In [ ]:
#path = DATASETS / "CeedLateralIntensiveData.csv"
path = DATASETS / "CeedEveron.csv"
data_storage = LogReaderV2(path)

In [3]:
data_storage.add_batch( "vx", use_jax_interp=True)
data_storage.add_batch( "vy", use_jax_interp=False)
data_storage.add_batch( "ay", use_jax_interp=False)
data_storage.add_batch( "yaw_rate",  use_jax_interp=False)
data_storage.add_batch( "steer", use_jax_interp=True)
data_storage.process_all()

Добавлен в очередь vx из default: time [0.000, 279.980], values [0.000, 14.826]
Добавлен в очередь vy из default: time [0.000, 279.980], values [-0.136, 0.157]
Добавлен в очередь ay из default: time [0.000, 279.980], values [-3.367, 3.035]
Добавлен в очередь yaw_rate из default: time [0.000, 279.980], values [-0.419, 0.416]
Добавлен в очередь steer из default: time [0.000, 279.980], values [-4.520, 4.082]

Общий t0 = 0.0
  используем jax
Обработан vx: норм. время [0.000, 279.980]
Обработан vy: норм. время [0.000, 279.980]
Обработан ay: норм. время [0.000, 279.980]
Обработан yaw_rate: норм. время [0.000, 279.980]
  используем jax
Обработан steer: норм. время [0.000, 279.980]
Общий временной интервал: [0.000, 279.980]


In [4]:
f_vx = data_storage.get_f_interp("vx")
f_vy = data_storage.get_f_interp("vy")
f_ay = data_storage.get_f_interp("ay")
f_steer = data_storage.get_f_interp("steer")
f_yaw_rate = data_storage.get_f_interp("yaw_rate")
t = data_storage.get_time("vx")
t1 = t[0]
t2 = t[-1]
t = np.linspace(t1, t2 - 10, 8000)
plt.plot(t, f_vx(t))
# plt.plot(t, f_vy(t))
#plt.plot(t, np.rad2deg(f_steer(t)))
# plt.plot(t, f_yaw_rate(t))

<Figure size 640x480 with 1 Axes>

In [5]:
fig = plt.figure(figsize=(20, 15))

plt.plot(t, f_ay(t))
plt.plot(t, f_vx(t)*f_vx(t)*np.tan(f_steer(t)/13)/2.65)

<Figure size 2000x1500 with 1 Axes>

In [6]:
fig = plt.figure(figsize=(20, 15))
plt.plot(t, f_vx(t)*np.tan(f_steer(t)/14)/2.65)
plt.plot(t, f_yaw_rate(t))

<Figure size 2000x1500 with 1 Axes>

In [7]:


def get_input_signals(t):
    return [f_vx(t), f_steer(t)]      




class DynamicModelRearAxle(ODESystem):
    def __init__(self, m, wheelbase, GR = None,  g=9.81,):
        self.m = m
        self.wheelbase = wheelbase
        self.g = g
        np = 4
        self.GR = GR
        if(self.GR is None):
            np +=1
        # состояния:  wz, vy_rear
        super().__init__(2, np, 2)

    def get_lateral_forces(self, rwa, vx, vy_rear, wz, theta):
        Cf_norm, Cr_norm, a_rel = theta[0], theta[1], theta[2]
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        # Скорость центра масс через заднюю ось
        vy_cm = vy_rear + b * wz
        alpha_f = rwa - (vy_cm + a * wz) / vx
        alpha_r = -(vy_cm - b * wz) / vx  
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr, a, b

    def get_derivative(self, state, params, u):
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        
        GR = self.GR
        if(self.GR is None):
            GR = params[4]
        
        rwa = steering / GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # динамика центра масс (необходима для сил и ускорения)
        vy_cm = vy_rear + b * wz
        dvy_cm = (Fyf + Fyr) / self.m - vx * wz
        dwz = (a * Fyf - b * Fyr) / Iz

        # производная vy_rear
        dvy_rear = dvy_cm - b * dwz
        return ca.vertcat(dwz, dvy_rear)
    
    def calc_acc(self, state, params, u, d=0.0):
        """
        Вычисляет поперечное ускорение в точке, смещённой на d от центра масс.
        d > 0 – вперёд, к передней оси; d < 0 – назад.
        Состояние state (SX): [tau, psi, wz, vy_rear, rwa, rwa_dot, ...]
        """
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        GR = self.GR
        if(self.GR is None):
            GR = params[4]
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # Ускорение центра масс
        a_lat_cm = (Fyf + Fyr) / self.m

        # Угловое ускорение
        dwz = (a * Fyf - b * Fyr) / Iz

        # Ускорение в заданной точке
        a_lat = a_lat_cm + d * dwz

        return a_lat
    
    def observation(self, state: SX, theta: SX, u: SX):
        a_rel = theta[2]
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        a_lat = self.calc_acc(state, theta, u, d = -b)
        wz, vy = state[0], state[1]
        return ca.vertcat(a_lat, wz)




config_dyn = {
    "class": DynamicModelRearAxle,
    "args": [1900, 2.65, None],                                 # wheelbase
    "c0": np.array([0.0]),      
    "theta_true": np.array([5,  5,  0.5,  0.4, 12]), #
    "delta_theta": 0*np.array([4, 4.0, 0.2, 0.24]),
    "input_signal": get_input_signals,  #vx steering
}

t_batches = [t]


measured_batches = [np.vstack((f_ay(t), f_yaw_rate(t))).T]
system, c0, theta_init, _ = create_system(config_dyn)



In [8]:
system.get_input_signals(0)
system.m

1900

In [9]:
measured_batches[0]

array([[-0.16414366, -0.04022771],
       [-0.38535924, -0.0362372 ],
       [-0.2247545 , -0.0283731 ],
       ...,
       [-0.21812041, -0.00222907],
       [ 0.06583901, -0.00208799],
       [-0.27200491, -0.002155  ]], shape=(8000, 2))

In [10]:
plt.plot(measured_batches[0][:140, :])

<Figure size 640x480 with 1 Axes>

In [11]:
gamma = np.zeros(2)
for i in range(2):
    gamma[i] = 1/np.std(measured_batches[0][:125, i])
    print(gamma[i])

6.439079179411109
145.63957212850372


In [12]:
# optimization_config.py (исправленная версия с доверительными интервалами)
import numpy as np
import matplotlib.pyplot as plt

class OptimizationConfig:
    """Конфигурация задачи. mu и lambda больше не задаются руками —
    адаптивный цикл подбирает их сам (см. adaptive_regularization.ipynb)."""
    def __init__(self):
        self.gamma = gamma
        #assert len(self.gamma) == n_obs
        self.n_iter = 30                   # верхняя граница итераций
        self.c0_cost = 1.0                  # вес начальной точки в интервале
        self.n_shoot = 10
        self.n_sub = 1   # элементов коллокации на интервал сетки (Colloc*-классы)

def setup_problem(system:ODESystem, 
                  problem_class, 
                  config,
                  state_measured_batches: np.array,
                  t_eval_batches: np.array):
    kwargs = {}
    if issubclass(problem_class, CollocationShooting):
        # свои параметры интегратора: n_sub, K, newton_tol, rootfinder_options...
        kwargs['n_sub'] = config.n_sub
    else:
        # ВАЖНО: без use_jax=True MultipleShooting интегрирует шуты
        # последовательным scipy-циклом — на 8000 точках это в разы медленнее
        # батчевого JAX
        kwargs['use_jax'] = True

    problem = problem_class(system, N_shoot=config.n_shoot, gamma=config.gamma,
                               c0_cost=config.c0_cost, **kwargs)
    for state_meas, t_meas in zip(state_measured_batches, t_eval_batches):
        problem.add_batch(state_meas, t_meas)
    return problem


In [13]:
from gauss_newton.adaptive import run_optimization_adaptive

In [14]:
from gauss_newton.normal_equations import (MultipleShootingAccum,
                                           CollocationShootingAccum)

# Четыре взаимозаменяемых класса задачи (что внутри — безразлично циклу):
#   MultipleShooting         — вариационные уравнения, JAX (явный интегратор)
#   CollocationShooting      — коллокации Радо IIA (жёсткие системы)
#   MultipleShootingAccum    — то же + H и g копятся сразу, большая J не строится
#   CollocationShootingAccum — коллокации + накопление H и g
problem_class = CollocationShootingAccum

config = OptimizationConfig()
problem = setup_problem(system, problem_class, config,
                        measured_batches, t_batches)
theta_full = problem.make_full_theta(theta_init, n_iter=10)

# Новый цикл: mu и lambda подбираются сами, остановка автоматическая.
# ВНИМАНИЕ: возвращает (theta_full, hist), а НЕ кортеж из 6 элементов.
theta_full, hist = run_optimization_adaptive(problem, theta_full,
                                             n_iter=config.n_iter, verbose=True)

# имена, которые используют ячейки ниже (plot_solution, theta_hist[-1], ...)
theta_hist = hist['theta']
r_meas_hist = hist['r_meas']
r_cont_hist = hist['r_cont']
ci_low_hist = hist['ci_low']
ci_high_hist = hist['ci_high']

# Старый цикл с ручными mu/mu_dec/lambda удалён из библиотеки:
# в ней остался один цикл (run_optimization_adaptive) и один шаг
# (gn_step). Воспроизведение прежней схемы — в adaptive_regularization.ipynb.


Iter   0 | accept rho= 0.938 | cost 7.683e+04 -> 3.523e+04 | mu 1.06e-05 | lam 3.33e-04
Iter   1 | reject rho=-344096812799517175796094763261242896361992008810215341791060509289203644760064.000 | lam 6.67e-04
Iter   2 | reject rho=-295274629287072360650844941387145797892884446815907341084267642880.000 | lam 2.67e-03
Iter   3 | reject rho=-35.457 | lam 2.13e-02
Iter   4 | accept rho= 0.831 | cost 3.523e+04 -> 3.350e+04 | mu 5.30e-06 | lam 1.51e-02
Iter   5 | accept rho= 0.929 | cost 3.350e+04 -> 3.321e+04 | mu 5.30e-06 | lam 5.61e-03
Iter   6 | accept rho= 0.243 | cost 3.321e+04 -> 3.320e+04 | mu 2.65e-06 | lam 6.37e-03
Iter   7 | accept rho= 1.067 | cost 3.320e+04 -> 3.320e+04 | mu 1.32e-06 | lam 2.12e-03
Iter   8 | accept rho= 0.525 | cost 3.320e+04 -> 3.320e+04 | mu 6.62e-07 | lam 2.12e-03
Iter   9 | accept rho= 1.599 | cost 3.320e+04 -> 3.320e+04 | mu 6.62e-07 | lam 7.07e-04
Iter  10 | reject rho=-1.410 | lam 1.41e-03
Iter  11 | reject rho=-6.377 | lam 5.66e-03
Iter  12 | reject rho

In [17]:

plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=0,
    plot_theta=True,
    plot_trajectory=1,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    r_meas_hist=r_meas_hist,
    r_cont_hist=r_cont_hist,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    #param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.n_theta)]
)

In [ ]:
# =====================================================================
# Сравнение четырёх классов задачи на этих же данных.
#
# Раньше здесь сравнивались «старый цикл против нового». Старого цикла
# (run_optimization с ручными mu/mu_dec/lambda) в библиотеке больше нет —
# остался один run_optimization_adaptive, поэтому варианты теперь
# отличаются только интегратором и способом получения H и g.
#
# Все имена с префиксом bench_ — переменные ноутбука (problem, config,
# theta_hist, ...) не затрагиваются, ячейку можно перезапускать отдельно.
# =====================================================================
import time

import numpy as np
import pandas as pd

from gauss_newton.problem import MultipleShooting
from gauss_newton.collocation_shooting import CollocationShooting
from gauss_newton.normal_equations import (MultipleShootingAccum,
                                           CollocationShootingAccum)
from gauss_newton.adaptive import run_optimization_adaptive

BENCH_VARIANTS = [
    ('1. MS (вариационные)',      MultipleShooting),
    ('2. Colloc (Радо IIA)',      CollocationShooting),
    ('3. MS + накопление H,g',    MultipleShootingAccum),
    ('4. Colloc + накопление',    CollocationShootingAccum),
]
BENCH_BASELINE = '1. MS (вариационные)'   # с чем сравниваем
BENCH_N_REPEAT = 3     # замеров стоимости одного пересчёта
BENCH_N_ITER = 30      # верхняя граница итераций


def bench_evaluate(bench_prob, bench_theta):
    """Один пересчёт: (H, g) накоплением или (J, R) — что задача умеет."""
    if hasattr(bench_prob, 'normal_equations'):
        return bench_prob.normal_equations(bench_theta).cost()
    _, bench_R, _, bench_RG = bench_prob.solve(bench_theta)
    return float(bench_R @ bench_R + bench_RG @ bench_RG)


def bench_instrument(bench_prob):
    """Логирует (время, стоимость) каждого пересчёта внутри цикла."""
    bench_log = []
    if hasattr(bench_prob, 'normal_equations'):
        bench_orig = bench_prob.normal_equations

        def bench_wrapped(bench_theta):
            bench_ne = bench_orig(bench_theta)
            bench_log.append((time.perf_counter(), bench_ne.cost()))
            return bench_ne
        bench_prob.normal_equations = bench_wrapped
    else:
        bench_orig = bench_prob.solve

        def bench_wrapped(bench_theta):
            bench_J, bench_R, bench_JG, bench_RG = bench_orig(bench_theta)
            bench_log.append((time.perf_counter(),
                              float(bench_R @ bench_R + bench_RG @ bench_RG)))
            return bench_J, bench_R, bench_JG, bench_RG
        bench_prob.solve = bench_wrapped
    return bench_log


bench_runs = []
for bench_label, bench_cls in BENCH_VARIANTS:
    # --- разовые издержки: сборка задачи + первый пересчёт (JIT/компиляция)
    bench_system, _, _, _ = create_system(config_dyn)   # свой CasADi-граф
    bench_cfg = OptimizationConfig()
    bench_cfg.n_iter = BENCH_N_ITER
    bench_t0 = time.perf_counter()
    bench_prob = setup_problem(bench_system, bench_cls, bench_cfg,
                               measured_batches, t_batches)
    bench_theta0 = bench_prob.make_full_theta(theta_init, n_iter=10)
    bench_evaluate(bench_prob, bench_theta0)
    bench_warmup = time.perf_counter() - bench_t0

    # --- стоимость одного пересчёта в стартовой точке (медиана)
    bench_times = []
    for _ in range(BENCH_N_REPEAT):
        bench_t0 = time.perf_counter()
        bench_evaluate(bench_prob, bench_theta0)
        bench_times.append(time.perf_counter() - bench_t0)
    bench_per_eval = float(np.median(bench_times))

    # --- полный прогон
    bench_log = bench_instrument(bench_prob)
    bench_error = None
    bench_start = time.perf_counter()
    try:
        bench_theta_opt, bench_hist = run_optimization_adaptive(
            bench_prob, bench_theta0, n_iter=BENCH_N_ITER)
        bench_n_iter = len(bench_hist['mu']) - 1
        bench_rejects = bench_n_iter - len(bench_hist['accepted'])
    except RuntimeError as bench_exc:   # напр. Ньютон коллокаций не сошёлся
        bench_error = str(bench_exc).splitlines()[0]
        bench_theta_opt, bench_n_iter, bench_rejects = bench_theta0, 0, 0
    bench_wall = time.perf_counter() - bench_start

    bench_runs.append(dict(
        label=bench_label, warmup=bench_warmup, per_eval=bench_per_eval,
        wall=bench_wall, n_iter=bench_n_iter, n_eval=len(bench_log),
        rejects=bench_rejects, theta=bench_theta_opt[:len(theta_init)],
        cost=bench_log[-1][1] if bench_log else np.nan,
        trace=[(bench_t - bench_start, bench_c) for bench_t, bench_c in bench_log],
        error=bench_error))
    print(f'{bench_label:<26} прогон {bench_wall:5.1f} с  ({bench_n_iter} итер)  '
          f'cost={bench_runs[-1]["cost"]:.4e}'
          + (f'  ОШИБКА: {bench_error}' if bench_error else ''))

# --- время до общей планки: 1% выше лучшей достигнутой стоимости
bench_bar = 1.01 * min(bench_r['cost'] for bench_r in bench_runs)
bench_base = next(bench_r['wall'] for bench_r in bench_runs
                  if bench_r['label'] == BENCH_BASELINE)
for bench_r in bench_runs:
    bench_hit = [bench_t for bench_t, bench_c in bench_r['trace'] if bench_c <= bench_bar]
    bench_r['to_bar'] = f'{bench_hit[0]:.1f} с' if bench_hit else 'не достиг'

bench_table = pd.DataFrame([dict(
    вариант=bench_r['label'],
    сборка=f'{bench_r["warmup"]:.1f} с',
    пересчёт=f'{bench_r["per_eval"]:.3f} с',
    прогон=f'{bench_r["wall"]:.1f} с',
    отн=f'{bench_r["wall"] / bench_base:.2f}x',
    итераций=bench_r['n_iter'],
    пересчётов=bench_r['n_eval'],
    отказов=bench_r['rejects'],
    до_планки=bench_r['to_bar'],
    стоимость=f'{bench_r["cost"]:.4e}',
    theta=np.array2string(bench_r['theta'], precision=2),
) for bench_r in bench_runs])

print(f'\nПланка: cost <= {bench_bar:.4e} (1% выше лучшей из достигнутых)')
print(f'Колонка «отн» — к варианту «{BENCH_BASELINE}».')
print('Абсолютные времена зависят от загрузки машины (разброс до ~1.5x между '
      'запусками), отношения между вариантами устойчивы.\n')
print(bench_table.to_string(index=False))


In [21]:
theta_full

array([ 9.88194274e+00,  4.33009733e+00,  2.27350709e-01,  1.21362478e-01,
       -1.04441483e-02, -2.29573476e-03,  2.73933588e-02, -2.03532206e-01,
       -3.78710664e-02,  1.76619606e-01,  8.23287392e-02, -2.58591144e-01,
        6.83927685e-02, -1.52109409e-01,  1.61231389e-01, -2.22290837e-01,
       -1.11889020e-01,  1.90053700e-01,  4.76493722e-02, -1.32532201e-02,
       -6.79276882e-02,  6.93703597e-02,  4.89501255e-02, -2.54830386e-02])

In [22]:
theta_hist[-1]


array([ 9.88194274e+00,  4.33009733e+00,  2.27350709e-01,  1.21362478e-01,
       -1.04441483e-02, -2.29573476e-03,  2.73933588e-02, -2.03532206e-01,
       -3.78710664e-02,  1.76619606e-01,  8.23287392e-02, -2.58591144e-01,
        6.83927685e-02, -1.52109409e-01,  1.61231389e-01, -2.22290837e-01,
       -1.11889020e-01,  1.90053700e-01,  4.76493722e-02, -1.32532201e-02,
       -6.79276882e-02,  6.93703597e-02,  4.89501255e-02, -2.54830386e-02])

In [25]:

plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    r_meas_hist=r_meas_hist,
    r_cont_hist=r_cont_hist,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    #param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.n_theta)]
)

In [16]:
config = theta_to_physical(theta_hist[-1][:5], 1900, 2.65)
theta_hist[-1][:5], config  # without GR estiname everon GR = 14


(array([ 6.49719488e+00,  1.97219412e-01,  2.44209539e-02,  7.86649727e-03,
        -3.79128332e-03]),
 {'Cf': np.float64(121101.21541773588),
  'Cr': np.float64(3675.972628410733),
  'a': np.float64(0.06471552779661062),
  'b': np.float64(2.5852844722033894),
  'Iz': np.float64(104.96070640273115),
  'steering_ratio': np.float64(-0.0037912833177193885)})

GR=14

In [70]:
config = theta_to_physical(theta_hist[-1][:5], 1900, 2.65)
theta_hist[-1][:5], config  # without GR estiname everon GR = 14


(array([ 5.15879858,  6.42911574,  0.48127906,  0.16365824, -0.04022771]),
 {'Cf': np.float64(96154.84673262002),
  'Cr': np.float64(119832.28827786002),
  'a': np.float64(1.2753895089999998),
  'b': np.float64(1.374610491),
  'Iz': np.float64(2183.65098176),
  'steering_ratio': np.float64(-0.0402277109618861)})

In [84]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:4], config  # without GR estiname poligon GR = 14


(array([5.20921863, 6.59715055, 0.48416129, 0.16590168]),
 {'Cf': np.float64(97094.62612658725),
  'Cr': np.float64(122964.28909468552),
  'a': np.float64(1.2830274062575469),
  'b': np.float64(1.366972593742453),
  'Iz': np.float64(2213.5847068342473)})

GR is indentified

In [32]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:5], config  # with GR estiname everon 

(array([ 2.41435822,  2.89092052,  0.42223067,  0.05247475, 12.48189028]),
 {'Cf': np.float64(45001.22293241562),
  'Cr': np.float64(53883.86765105967),
  'a': np.float64(1.1189112855992658),
  'b': np.float64(1.5310887144007341),
  'Iz': np.float64(700.1575068630275)})

In [16]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:5], config # # with GR estiname poligon 


(array([ 4.98018613,  6.53100463,  0.47473288,  0.17420062, 13.04099914]),
 {'Cf': np.float64(92825.68929087718),
  'Cr': np.float64(121731.39529265452),
  'a': np.float64(1.2580421264472461),
  'b': np.float64(1.3919578735527538),
  'Iz': np.float64(2324.3153172115185)})

In [ ]:
# Как выглядит решение при заданных вручную параметрах (без оптимизации).
# Вектор обрезается под текущую конфигурацию: с GR=None параметров 5
# (последний — передаточное отношение), с фиксированным GR — 4.
theta_manual = np.array([5.15879858, 6.42911574, 0.48127906, 0.16365824,
                         12.0])[:system.n_theta]

# Имя config_manual, а не config: ниже по ноутбуку `config` — это словарь
# физических параметров от theta_to_physical, и его нельзя затирать
config_manual = OptimizationConfig()
problem = setup_problem(system, problem_class, config_manual,
                    measured_batches,  t_batches)
theta_full = problem.make_full_theta(theta_manual, n_iter = 10)
theta_hist = [theta_full]


plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=0,
    plot_measurements = 1,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    #param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.n_theta)]
)

In [ ]:
(1900/2.65)*(config["a"]/config['Cf'] - config["b"]/config['Cr'])

np.float64(-0.0037202467327500568)

In [ ]:
(1900/2.65)*(1.2333664515275726 /59406.34085264137- 1.4166335484724273/133076.911608564)

0.007253199525996578

In [ ]:
0.007253199525996578*25**2

4.533249703747861

In [ ]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
config


{'Cf': np.float64(59406.34085264137),
 'Cr': np.float64(133076.911608564),
 'a': np.float64(1.2333664515275726),
 'b': np.float64(1.4166335484724273),
 'Iz': np.float64(1092.2923296101428)}

In [ ]:
theta_hist[0][:5], theta_hist[-1][:5]

(array([ 4.49955229,  7.92448103,  0.54657406,  0.16234435, 12.816658  ]),
 array([ 2.76587496,  8.19576105,  0.53505593,  0.08770295, 12.37426532]))

In [ ]:
theta_hist[0][:5], theta_hist[-1][:5]

(array([ 3.0642791 ,  7.26640357,  0.6031844 ,  0.31198512, 14.        ]),
 array([ 4.49955229,  7.92448103,  0.54657406,  0.16234435, 12.81665875]))

In [ ]:
config = theta_to_physical(theta_hist[-1][:5], 1700, 2.65)
config


{'Cf': np.float64(69550.88993644534),
 'Cr': np.float64(225649.61137671844),
 'a': np.float64(1.7656617581599081),
 'b': np.float64(0.8843382418400918),
 'Iz': np.float64(2649.4601012946537),
 'steering_ratio': np.float64(12.684747488766583)}

In [ ]:
# Физические параметры последней оценки: Cf, Cr, a, b, Iz (+ GR, если оценивался)
theta_to_physical(theta_hist[-1][:system.n_theta], 1900, 2.65)